# Notebook 13 — Statistical Significance Tests

## Purpose

Every result so far comes from a single seed (or fixed Q runs).
This notebook validates the key claims with proper statistical tests:

1. **Is MIGA Fr significantly lower than baselines under MAR?**
   Wilcoxon signed-rank test (one-sample): H₁: MIGA Fr < baseline Fr.
2. **Is MIGA Fr still lowest under MNAR?**
   The Haberman `top` MNAR finding (F7) — MIGA achieves lowest Fr but
   highest RMSE — requires confirmation that baselines have higher Fr.
3. **Effect sizes** — Cohen's d to quantify the Fr gap.

## Design

- N=10 independent MIGA seeds for each (dataset × mechanism) pair
- Baselines (Mean/KNN/MICE) are deterministic — single run each
- Wilcoxon one-sample test: MIGA Fr distribution vs. a constant (baseline value)
- Datasets: Iris, Glass, Haberman at 30% missing
- Mechanisms: MAR, top MNAR, tails MNAR

**Run scripts first (parallel):**
```
.venv/bin/python scripts/run_significance.py Iris
.venv/bin/python scripts/run_significance.py Glass
.venv/bin/python scripts/run_significance.py Haberman
```
Each saves `results/13_significance_<dataset>.json`.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from miga import MIGA
from miga.fitness import FitnessEvaluator
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance
from miga.data_utils import apply_mar, apply_mnar, auto_generators, compute_metrics, EXCLUDE_COLS
from miga.paper_results import (
    TABLE3_PARAMS, BENCHMARK_Q,
    TABLE4_RMSE, TABLE5_MAD, TABLE6_COD,
    METHODS, PERCENTAGES,
)

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Load Results

In [ ]:
from scipy import stats

DATASET_NAMES = ["Iris", "Glass", "Haberman"]
MECHANISMS    = ["mar", "top", "tails"]
METHODS       = ["Mean", "KNN", "MICE"]

sig_data = {}
for ds in DATASET_NAMES:
    path = os.path.join(RESULTS_DIR, f"13_significance_{ds.lower()}.json")
    if not os.path.exists(path):
        print(f"  [MISSING] {path}")
        continue
    with open(path) as f:
        data = json.load(f)
    sig_data[ds] = data[ds]
    print(f"  Loaded: {ds}")
print(f"Loaded {len(sig_data)}/{len(DATASET_NAMES)} datasets.")

## 2. Fr: MIGA vs Baselines under MAR (Wilcoxon Tests)

In [ ]:
if sig_data:
    print("=" * 70)
    print("Fr under MAR — MIGA (10 seeds) vs deterministic baselines")
    print("H1: MIGA Fr < baseline Fr  (one-sided Wilcoxon)")
    print("=" * 70)
    print()
    print(f"  {'Dataset':<10}  {'Baseline':<6}  {'MIGA mean Fr':>13}  {'Baseline Fr':>12}  {'p-value':>9}  {'Sig?':>6}  {'Win?'}")
    print("  " + "-" * 70)

    for ds in DATASET_NAMES:
        if ds not in sig_data:
            continue
        mech_data = sig_data[ds].get("mar", {})
        miga_frs = [f for f in mech_data.get("miga_fr", []) if f == f]
        if not miga_frs:
            continue
        miga_arr = np.array(miga_frs)

        for bl_name in METHODS:
            bl_fr = mech_data.get("baselines", {}).get(bl_name, {}).get("Fr")
            if bl_fr is None or bl_fr != bl_fr:
                continue
            w_data = mech_data.get("wilcoxon_fr", {}).get(bl_name, {})
            p     = w_data.get("p", float("nan"))
            miga_wins = miga_arr.mean() < bl_fr
            sig   = "✓✓" if p < 0.01 else ("✓" if p < 0.05 else ("~" if p < 0.10 else "n.s."))
            win   = "MIGA<" if miga_wins else "MIGA>"
            print(f"  {ds:<10}  {bl_name:<6}  {miga_arr.mean():>13.4f}  {bl_fr:>12.4f}  {p:>9.3f}  {sig:>6}  {win}")

## 3. Fr under MNAR — The Key Finding

In [ ]:
if sig_data:
    print("=" * 70)
    print("Fr under MNAR — MIGA (10 seeds) vs baselines")
    print("Critical test: does MIGA still achieve lowest Fr under MNAR?")
    print("=" * 70)

    for mech in ["top", "tails"]:
        print(f"\n--- {mech.upper()} MNAR ---")
        print(f"  {'Dataset':<10}  {'MIGA Fr':>10}  {'Mean Fr':>10}  {'KNN Fr':>10}  {'MICE Fr':>10}  {'MIGA rank':>10}  {'MIGA RMSE':>10}")
        print("  " + "-" * 75)

        for ds in DATASET_NAMES:
            if ds not in sig_data:
                continue
            mech_data = sig_data[ds].get(mech, {})
            if not mech_data:
                continue
            miga_frs  = [f for f in mech_data.get("miga_fr", []) if f == f]
            miga_rmse = np.mean(mech_data.get("miga_rmse", [float("nan")]))
            if not miga_frs:
                continue
            miga_mean = np.mean(miga_frs)
            bls = mech_data.get("baselines", {})
            bl_frs = {m: (bls.get(m, {}).get("Fr") or float("nan")) for m in METHODS}

            all_frs = {"MIGA": miga_mean, **bl_frs}
            ranked  = sorted(all_frs, key=lambda k: all_frs[k] if all_frs[k]==all_frs[k] else float("inf"))
            rank    = ranked.index("MIGA") + 1

            print(f"  {ds:<10}  {miga_mean:>10.4f}  "
                  f"{bl_frs['Mean']:>10.4f}  {bl_frs['KNN']:>10.4f}  "
                  f"{bl_frs['MICE']:>10.4f}  {rank:>10}/4  {miga_rmse:>10.4f}")

## 4. The Complete Fr vs RMSE Picture

This table captures the central empirical finding of the thesis.

In [ ]:
if sig_data:
    print("=" * 80)
    print("MIGA: lower Fr (better distributional fit) vs MICE: lower RMSE (better pointwise accuracy)")
    print("=" * 80)
    print()
    for ds in DATASET_NAMES:
        if ds not in sig_data:
            continue
        for mech in MECHANISMS:
            mech_data = sig_data[ds].get(mech, {})
            if not mech_data:
                continue
            miga_frs  = [f for f in mech_data.get("miga_fr",  []) if f == f]
            miga_rmse = np.mean(mech_data.get("miga_rmse", [float("nan")]))
            if not miga_frs:
                continue
            miga_fr_mean = np.mean(miga_frs)
            bls = mech_data.get("baselines", {})
            mice_fr   = bls.get("MICE", {}).get("Fr")   or float("nan")
            mice_rmse = bls.get("MICE", {}).get("rmse") or float("nan")
            knn_fr    = bls.get("KNN",  {}).get("Fr")   or float("nan")
            knn_rmse  = bls.get("KNN",  {}).get("rmse") or float("nan")

            fr_ratio   = mice_fr   / miga_fr_mean if miga_fr_mean and miga_fr_mean==miga_fr_mean else float("nan")
            rmse_ratio = miga_rmse / mice_rmse    if mice_rmse and mice_rmse==mice_rmse else float("nan")

            print(f"  {ds:<10}  {mech:<7}  "
                  f"MIGA Fr={miga_fr_mean:.4f}  MICE Fr={mice_fr:.4f}  "
                  f"[MIGA Fr {fr_ratio:.2f}x better]  |  "
                  f"MIGA RMSE={miga_rmse:.4f}  MICE RMSE={mice_rmse:.4f}  "
                  f"[MICE RMSE {rmse_ratio:.2f}x better]")

## 5. Visualisation — Fr and RMSE across Seeds (Boxplots)

In [ ]:
if sig_data:
    n_ds = len(sig_data)
    fig, axes = plt.subplots(2, n_ds, figsize=(5 * n_ds, 8))
    if n_ds == 1:
        axes = axes.reshape(2, 1)

    mech_colours = {"mar": "tab:blue", "top": "tab:red", "tails": "tab:purple"}

    for col, ds in enumerate(ds for ds in DATASET_NAMES if ds in sig_data):
        ds_data = sig_data[ds]

        # Fr plot
        ax_fr = axes[0, col]
        all_frs = []
        labels  = []
        for mech in MECHANISMS:
            mech_data = ds_data.get(mech, {})
            frs = [f for f in mech_data.get("miga_fr", []) if f == f]
            if frs:
                all_frs.append(frs)
                labels.append(f"MIGA\n{mech}")
            # baseline Fr as horizontal lines
            for bl_name in ["MICE"]:
                bl_fr = mech_data.get("baselines", {}).get(bl_name, {}).get("Fr")
                if bl_fr and bl_fr == bl_fr:
                    ax_fr.axhline(bl_fr, color=mech_colours[mech], ls="--",
                                  alpha=0.6, label=f"MICE Fr ({mech})")

        if all_frs:
            bp = ax_fr.boxplot(all_frs, labels=labels, patch_artist=True)
            for patch, mech in zip(bp["boxes"], MECHANISMS):
                patch.set_facecolor(mech_colours[mech])
                patch.set_alpha(0.6)
        ax_fr.set_title(f"{ds} — Fr (10 seeds)")
        ax_fr.set_ylabel("Fr (lower = better distributional fit)")
        ax_fr.grid(axis="y", alpha=0.3)

        # RMSE plot
        ax_rmse = axes[1, col]
        all_rmse = []
        labels2  = []
        for mech in MECHANISMS:
            mech_data = ds_data.get(mech, {})
            rmse_vals = mech_data.get("miga_rmse", [])
            if rmse_vals:
                all_rmse.append(rmse_vals)
                labels2.append(f"MIGA\n{mech}")
            for bl_name in ["MICE"]:
                bl_rmse = mech_data.get("baselines", {}).get(bl_name, {}).get("rmse")
                if bl_rmse:
                    ax_rmse.axhline(bl_rmse, color=mech_colours[mech], ls="--",
                                    alpha=0.6, label=f"MICE RMSE ({mech})")

        if all_rmse:
            bp = ax_rmse.boxplot(all_rmse, labels=labels2, patch_artist=True)
            for patch, mech in zip(bp["boxes"], MECHANISMS):
                patch.set_facecolor(mech_colours[mech])
                patch.set_alpha(0.6)
        ax_rmse.set_title(f"{ds} — RMSE (10 seeds)")
        ax_rmse.set_ylabel("NRMSE (lower = better pointwise accuracy)")
        ax_rmse.grid(axis="y", alpha=0.3)

    plt.suptitle("MIGA (10 seeds) vs MICE (dashed): Fr and RMSE across MAR / top MNAR / tails MNAR",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "13_fr_rmse_boxplots.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/13_fr_rmse_boxplots.png")

## 6. Summary Table for Thesis

In [ ]:
if sig_data:
    print("=" * 80)
    print("THESIS TABLE: MIGA Fr and RMSE vs MICE Fr and RMSE (mean ± std over 10 seeds)")
    print("=" * 80)
    print()
    print(f"  {'Dataset':<10}  {'Mech':<7}  {'MIGA Fr':>18}  {'MICE Fr':>10}  {'MIGA RMSE':>18}  {'MICE RMSE':>10}  {'Sig (Fr)':>10}")
    print("  " + "-" * 90)

    for ds in DATASET_NAMES:
        if ds not in sig_data:
            continue
        for mech in MECHANISMS:
            mech_data = sig_data[ds].get(mech, {})
            if not mech_data:
                continue
            miga_frs  = np.array([f for f in mech_data.get("miga_fr", []) if f == f])
            miga_rmse = np.array(mech_data.get("miga_rmse", []))
            bls       = mech_data.get("baselines", {})
            mice_fr   = bls.get("MICE", {}).get("Fr")   or float("nan")
            mice_rmse = bls.get("MICE", {}).get("rmse") or float("nan")
            w_data    = mech_data.get("wilcoxon_fr", {}).get("MICE", {})
            p         = w_data.get("p", float("nan"))
            sig       = "p<0.01**" if p < 0.01 else ("p<0.05*" if p < 0.05 else ("p<0.10~" if p < 0.10 else "n.s."))

            if len(miga_frs) and len(miga_rmse):
                fr_str   = f"{miga_frs.mean():.3f}±{miga_frs.std():.3f}"
                rmse_str = f"{miga_rmse.mean():.3f}±{miga_rmse.std():.3f}"
            else:
                fr_str = rmse_str = "n/a"

            print(f"  {ds:<10}  {mech:<7}  {fr_str:>18}  {mice_fr:>10.4f}  {rmse_str:>18}  {mice_rmse:>10.4f}  {sig:>10}")